# Toy bilevel problem — conditioning sweep

Compares **PABLO**, **BOME**, **TSP**, **STABLE**, **AAL**, and **PBGD-Free** as the conditioning
constant $c$ varies.

$$F(w,\theta) = (w+1)^4 - 2(w+1)^2 + \tfrac{1}{c}(\theta-\theta^2)^2 + 1 \qquad \text{(upper)}$$
$$J(w,\theta) = (w-1)^4 - 2(w-1)^2 + \tfrac{1}{2c}\theta^2 + 1 \qquad \text{(lower)}$$

**Protocol**
1. *Tuning phase* (4×4 init grid): for each $(c, \text{method})$, try five multiplicative
   step-size scales $\{0.1\times, 0.3\times, 1\times, 3\times, 10\times\}$ relative to the
   $c=10$ baseline hyperparameters. Pick the scale with the lowest median final loss
   $\bar{F}_{T} + \bar{J}_{T}$ across starts.
2. *Full sweep* (10×10 init grid): run all methods with their tuned hyperparameters.
3. **Convergence metric**: first iteration $t$ at which the *median* (across all starting
   points) upper-level loss $F$ **and** lower-level loss $J$ are both $< 10^{-9}$.
   Blank = not reached within `N_ITERS` iterations.
4. Save a convergence CSV + a hyperparameters JSON.

In [ ]:
import os, sys, json, csv, time
import numpy as np

sys.path.insert(0, "experiments")
import toy_lib
from toy_lib import run_sweep, summarise, summary_table, theta_error, W_STAR

%load_ext autoreload
%autoreload 2

print("methods available in toy_lib:", toy_lib.METHODS)
print("cpu count:", os.cpu_count())

## Settings — edit here

`HPARAMS_BASE` holds the $c=10$ tuned values for PABLO/BOME/TSP/STABLE (from
`Exp_ToyProblem_InitSweep`) and the code defaults for AAL/PBGD-Free (not previously tuned).
`STEP_KEYS` lists which keys are treated as step-size parameters for each method —
only these are scaled during the tuning search.

In [ ]:
C_VALUES = [0.1, 1.0, 10.0]   # conditioning values to sweep
METHODS  = ["PABLO", "BOME", "TSP", "STABLE", "AAL", "PBGD-Free"]

# ---- baseline hyperparameters (c = 10 starting point) --------------------------------
HPARAMS_BASE = {
    # PABLO: BLO_optimiser_PABLO_Prox_Bound
    "PABLO": dict(
        lr=0.003, alpha=0.9, eps=1e-3, rho=0.3,
        lr_lambda=0.0, tau=10.0, y_rate=0.5,
        grad_clip=10.0, lambda_bound=100.0, total_steps=None,
    ),
    # BOME
    "BOME": dict(
        lr=0.007, opt="adam", inner_steps=10, inner_lr=0.01,
        eta=0.1, barrier="grad_norm",
    ),
    # TSP (paper-faithful)
    "TSP": dict(
        gamma=0.5, delta=0.0, p=0.01, dual_tau=0.5,
        mu=0.5, theta=1.0, eta=0.2, step=0.1,
        omega=0.05, lambda_init=1.0,
    ),
    # STABLE
    "STABLE": dict(
        lr_w=0.05, lr_theta=0.5, tau=1.0, mu_g=0.001, C_gxy=10.0,
    ),
    # AAL — code defaults (not previously tuned for this problem)
    "AAL": dict(
        L_f=71.0, L_grad_g=71.0, L_H=60.0,
        gamma=1.0, rho0=0.0, alpha=1e-3,
        eps=1e-5, tau=1e-3, tau_decay=False,
    ),
    # PBGD-Free — code defaults (not previously tuned for this problem)
    "PBGD-Free": dict(
        gamma=10.0, eta=1e-2, eta_gamma=1e-2, K=1,
    ),
}

# ---- which keys to scale during the step-size search --------------------------------
STEP_KEYS = {
    "PABLO":     ["lr", "y_rate"],
    "BOME":      ["lr", "inner_lr"],
    "TSP":       ["step", "eta", "dual_tau"],
    "STABLE":    ["lr_w", "lr_theta"],
    "AAL":       ["alpha", "tau"],
    "PBGD-Free": ["eta", "eta_gamma"],
}

SCALES   = [0.1, 0.3, 1.0, 3.0, 10.0]

N_ITERS       = 2000
TUNE_N        = 4       # grid size for tuning (4×4 = 16 starts)
FULL_N        = 10      # grid size for full sweep (10×10 = 100 starts)
GRID_LO, GRID_HI = -1.5, 1.5
SEED          = 0
LOSS_TOL      = 1e-9
N_JOBS        = max(1, (os.cpu_count() or 2) - 1)

n_tune_runs = len(SCALES) * len(METHODS) * TUNE_N**2
n_full_runs = len(METHODS) * FULL_N**2
print(f"Tuning:     {n_tune_runs} runs × {N_ITERS} iters per c value")
print(f"Full sweep: {n_full_runs} runs × {N_ITERS} iters per c value")
print(f"c values: {C_VALUES}")

## Tuning phase

For each $(c, \text{method})$, runs five step-size scales on a 4×4 init grid and picks the one
minimising median final loss $\bar{F}_T + \bar{J}_T$. Results are stored in `all_hparams`.

In [ ]:
all_hparams = {}   # {c: {method: best_cfg}}
tuning_log  = {}   # {c: {method: [(scale, loss), ...]}}

for c in C_VALUES:
    print(f"\n{'='*65}")
    print(f"  TUNING  c = {c:g}")
    print(f"{'='*65}")
    tuned = {}
    log_c = {}

    for m in METHODS:
        log_m = []
        best_loss, best_scale, best_cfg = np.inf, 1.0, HPARAMS_BASE[m].copy()

        for s in SCALES:
            # Scale only the float step-size keys; booleans and ints are left unchanged
            cfg = {
                k: (v * s if (k in STEP_KEYS[m] and isinstance(v, float)) else v)
                for k, v in HPARAMS_BASE[m].items()
            }
            try:
                sw = run_sweep(
                    {m: cfg}, c=c, grid_lo=GRID_LO, grid_hi=GRID_HI,
                    grid_n=TUNE_N, n_iters=N_ITERS, seed=SEED,
                    n_jobs=N_JOBS, progress=False,
                )
                sm = summarise(sw)[m]
                if sm["median_curve_UL"] is None:
                    loss = np.inf
                else:
                    ul = float(sm["median_curve_UL"][-1])
                    ll = float(sm["median_curve_LL"][-1])
                    loss = (ul if np.isfinite(ul) else 1e30) + (ll if np.isfinite(ll) else 1e30)
            except Exception as exc:
                loss = np.inf
                print(f"    {m} s={s:.1f}x  ERROR: {exc}")

            log_m.append((s, loss))
            tag = f"{loss:.3e}" if np.isfinite(loss) else "diverged"
            print(f"    {m:<12}  scale={s:.1f}x  loss={tag}")
            if loss < best_loss:
                best_loss, best_scale, best_cfg = loss, s, cfg

        tuned[m] = best_cfg
        log_c[m] = log_m
        print(f"  -> {m}: best scale = {best_scale:.1f}x  (loss = {best_loss:.3e})\n")

    all_hparams[c] = tuned
    tuning_log[c]  = log_c

print("Tuning complete.")

## Full sweep — 10×10 init grid with tuned hyperparameters

In [ ]:
all_sweep   = {}   # {c: raw sweep output}
all_summary = {}   # {c: summarise output}

for c in C_VALUES:
    print(f"\n{'='*65}")
    print(f"  FULL SWEEP  c = {c:g}")
    print(f"{'='*65}")
    t0 = time.time()
    sw = run_sweep(
        all_hparams[c], c=c, grid_lo=GRID_LO, grid_hi=GRID_HI,
        grid_n=FULL_N, n_iters=N_ITERS, seed=SEED, n_jobs=N_JOBS,
    )
    all_sweep[c]   = sw
    all_summary[c] = summarise(sw)
    print(f"  done in {time.time()-t0:.1f}s")
    print(summary_table(all_summary[c], order=METHODS))

## Convergence table

First iteration $t$ at which median $F_t < 10^{-9}$ **and** median $J_t < 10^{-9}$
across all 100 starting points.

In [ ]:
iters_to_conv = {}   # {c: {method: int or nan}}

for c in C_VALUES:
    iters_to_conv[c] = {}
    for m in METHODS:
        sm = all_summary[c][m]
        if sm["median_curve_UL"] is None:
            iters_to_conv[c][m] = np.nan
            continue
        ul = sm["median_curve_UL"]
        ll = sm["median_curve_LL"]
        hit = np.where((ul < LOSS_TOL) & (ll < LOSS_TOL))[0]
        iters_to_conv[c][m] = int(hit[0] + 1) if len(hit) else np.nan

# Print convergence table
print(f"Iterations to reach median F < {LOSS_TOL:g} AND J < {LOSS_TOL:g}"
      f"  (— = not reached in {N_ITERS} iters)\n")
header = f"{'algorithm':<15}" + "".join(f"  c={c:>6g}" for c in C_VALUES)
print(header)
print("-" * len(header))
for m in METHODS:
    row = f"{m:<15}"
    for c in C_VALUES:
        v = iters_to_conv[c][m]
        row += f"  {'—' if np.isnan(v) else int(v):>8}"
    print(row)

## Save

Writes three files to `results/`:
* `*_iters.csv` — convergence table (rows = algorithms, columns = c values)
* `*_hparams.json` — tuned hyperparameters per (c, method)
* `*_detail.csv` — per-start final losses for every (c, method, w0, θ0)

In [ ]:
os.makedirs("results", exist_ok=True)
c_tag = "_".join(f"{c:g}" for c in C_VALUES)
tag   = f"cond_sweep_c{c_tag}_n{FULL_N}_it{N_ITERS}"

# 1. Convergence CSV
csv_path = f"results/{tag}_iters.csv"
with open(csv_path, "w", newline="") as fh:
    wr = csv.writer(fh)
    wr.writerow(["algorithm"] + [f"c={c:g}" for c in C_VALUES])
    for m in METHODS:
        wr.writerow([m] + [
            ("" if np.isnan(iters_to_conv[c][m]) else int(iters_to_conv[c][m]))
            for c in C_VALUES
        ])
print(f"Saved: {csv_path}")

# 2. Hyperparameters JSON
hp_path = f"results/{tag}_hparams.json"
with open(hp_path, "w") as fh:
    json.dump(
        {f"c={c:g}": {m: hp for m, hp in hps.items()} for c, hps in all_hparams.items()},
        fh, indent=2, default=str,
    )
print(f"Saved: {hp_path}")

# 3. Per-start detail CSV
detail_path = f"results/{tag}_detail.csv"
with open(detail_path, "w", newline="") as fh:
    wr = csv.writer(fh)
    wr.writerow(["c", "algorithm", "w0", "theta0",
                 "final_ul_loss", "final_ll_loss",
                 "final_w_err", "final_theta_err"])
    for c in C_VALUES:
        for m in METHODS:
            d = all_sweep[c][m]
            for (w0, t0), w, th, ul, ll in zip(
                    d["starts"], d["w"], d["theta"], d["ul_loss"], d["ll_loss"]):
                wr.writerow([
                    c, m, w0, t0,
                    float(ul[-1]), float(ll[-1]),
                    float(abs(w[-1] - W_STAR)),
                    float(theta_error(th[-1])),
                ])
print(f"Saved: {detail_path}")

print("\nDone.")